Notebook to create cross section as a function of column saturation fraction from WRF TC output for Haiyan and Maria.

Fig. 1 of paper

James Ruppert  
jruppert@ou.edu  
3/14/26

 #### Main settings

In [ ]:
import numpy as np
import matplotlib
from matplotlib import ticker, colors
import matplotlib.pyplot as plt
import pickle
import seaborn as sns

In [ ]:
# Index variable (2D; independent var)
ivar_select = 'sf'

# Fill variable (3D; dependent var)
fillvar_select = 'lwcrf'

# Contour variable (3D; dependent var)
contvar_select = 'w'

# Number of sample time steps
nt=24

 #### Additional settings and directories

In [ ]:
storm = 'haiyan'
# storm = 'maria'

data_main = "./data/"
figdir = "./figures/"

# Tests to read and compare
tests = ['ctl']
ntest=len(tests)

# Members
nmem = 10 # number of ensemble members (1-5 have NCRF)
# nmem = 2
enstag = str(nmem)

In [ ]:
# Ensemble member info
memb0=1 # Starting member to read
memb_nums=np.arange(memb0,nmem+memb0,1)
memb_nums_str=memb_nums.astype(str)
nustr = np.char.zfill(memb_nums_str, 2)
memb_all=np.char.add('memb_',nustr)

# Get dimensions
nz=39
pres = np.arange(1000,25,-25)
dp=(pres[0]-pres[1])*1e2

 #### Index aka Bin variable

In [ ]:
# Variable settings

# PW
if ivar_select == 'pw':
    # fmin=35;fmax=80 # mm
    # step=1
    # bins=np.arange(fmin,fmax+step,step)
    xlabel='Column water vapor [mm]'
    log_x='linear'
# Column saturation fraction
elif ivar_select == 'sf':
    # fmin=30;fmax=100 # %
    fmin=.3;fmax=1 # %
    # step=0.01
    # bins=np.arange(fmin,fmax+step,step)
    # xlabel='Column saturation fraction [%]'
    xlabel='Column saturation fraction'
    log_x='linear'
# Rainfall rate
elif ivar_select == 'rain':
    # bins=10.**(np.arange(1,8,0.3)-4)
    # bins=10.**(np.arange(0,8,0.3)-4)
    xlabel='Rainfall rate [mm/hr]'
    log_x='log'
# LW-ACRE
elif ivar_select == 'lwacre':
    # fmin=-50; fmax=200 # W/m2
    # step=5
    # bins=np.arange(fmin,fmax+step,step)
    xlabel='LW-ACRE [W/m$^2$]'
    log_x='linear'
# Vertical mass flux
elif ivar_select == 'vmf':
    # bins=10.**(np.arange(1,8,0.3)-3)
    # bins=np.flip(-1.*bins)
    xlabel='Vertical mass flux [kg/m/s]'
    log_x='log'
# Theta-e (equivalent potential temperature)
elif ivar_select == 'th_e':
    # fmin=315; fmax=365 # K
    # step=1
    # bins=np.arange(fmin,fmax+step,step)
    xlabel=r'$\theta_e$ [K]'
    log_x='linear'

# nbins = np.size(bins)

# Create axis of bin center-points for plotting
# bin_axis = (bins[np.arange(nbins-1)]+bins[np.arange(nbins-1)+1])/2

 ### Read from pickle file

In [ ]:
# Read one file to retrieve nbins, bin_axis before reading everything else
pickle_file = data_main+'binned_2d_'+tests[0]+'_'+ivar_select+'_'+str(nmem)+'memb_'+str(nt)+'hrs_'+storm+'.pkl'
with open(pickle_file, 'rb') as file:
    bins, frequency, icrf_binned, ihdia_binned, itmpk_binned, iqv_binned, icvar_binned, ilwacre_binned, ish_binned, ilh_binned, irain_binned, \
        istrat_binned, ilwacre_strat, irain_strat = pickle.load(file)

# Create axis of bin center-points for plotting
nbins = np.size(bins)
bin_axis = (bins[np.arange(nbins-1)]+bins[np.arange(nbins-1)+1])/2

# Begin reading all

# var_binned=np.ma.zeros((ntest,nbins-1,nz))
crf_binned=np.ma.zeros((ntest,nbins-1,nz))
hdia_binned=np.ma.zeros((ntest,nbins-1,nz))
tmpk_binned=np.ma.zeros((ntest,nbins-1,nz))
qv_binned=np.ma.zeros((ntest,nbins-1,nz))
cvar_binned=np.ma.zeros((ntest,nbins-1,nz))
strat_binned=np.ma.zeros((ntest,nbins-1,6)) # Bin count: 0-non-raining, 1-conv, 2-strat, 3-other/anvil
lwacre_binned=np.ma.zeros((ntest,nbins-1))
sh_binned=np.ma.zeros((ntest,nbins-1))
lh_binned=np.ma.zeros((ntest,nbins-1))
rain_binned=np.ma.zeros((ntest,nbins-1))
lwacre_strat=np.ma.zeros((ntest,nbins-1,6)) # LWACRE in: 0-non-raining, 1-conv, 2-strat, 3-other/anvil
rain_strat=np.ma.zeros((ntest,nbins-1,6)) # rain in: 0-non-raining, 1-conv, 2-strat, 3-other/anvil

for itest in range(ntest):
    pickle_file = data_main+'binned_2d_'+tests[itest]+'_'+ivar_select+'_'+str(nmem)+'memb_'+str(nt)+'hrs_'+storm+'.pkl'
    with open(pickle_file, 'rb') as file:
        bins, frequency, icrf_binned, ihdia_binned, itmpk_binned, iqv_binned, icvar_binned, ilwacre_binned, ish_binned, ilh_binned, irain_binned, \
            istrat_binned, ilwacre_strat, irain_strat = pickle.load(file)
    # var_binned[itest,...]    = ivar_binned
    crf_binned[itest,...]     = icrf_binned
    hdia_binned[itest,...]    = ihdia_binned
    convert = 1./(3600*24) # --> K/s to K/day
    tmpk_binned[itest,...]    = itmpk_binned*convert
    qv_binned[itest,...]      = iqv_binned*convert
    cvar_binned[itest,...]    = icvar_binned
    strat_binned[itest,...]   = istrat_binned
    lwacre_binned[itest,...]  = ilwacre_binned
    sh_binned[itest,...]      = ish_binned
    lh_binned[itest,...]      = ilh_binned
    rain_binned[itest,...]    = irain_binned
    lwacre_strat[itest,...]   = ilwacre_strat
    rain_strat[itest,...]     = irain_strat

---
## Plotting routines

In [ ]:
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 10}

matplotlib.rc('font', **font)

sns.set_theme(style="ticks", font_scale=1.2, rc={'xtick.bottom': True, 'ytick.left': True,
                                 "axes.spines.right": False, "axes.spines.top": False,})

### Figure settings

In [ ]:
# Three-dimensional dependent variables ("var")

# Radar Reflectivity
if fillvar_select == 'dbz':
    title = 'Ref'
    units_var1 = 'dBZ'
    cmin = -20; cmax=20
    cmap='RdBu_r'
# Radiation
elif fillvar_select == 'lwcrf':
    title = 'LW-CRF'
    figtag = fillvar_select
    cmap='RdBu_r'
    units_var1 = 'K/d'
    cmax=4; cmin=-1.*cmax
# Horizontal temperature anomaly
elif fillvar_select == 'tprm':
    title = r"$\theta_v'$"
    figtag = 'thprm'
    units_var1 = 'K'
    # cmax=1; cmin=-1.*cmax
    cmap='RdBu_r'
    cmax=2; cmin=-1.*cmax
    cmax_diff=.4; cmin_diff=-1.*cmax_diff
# Relative humidity
elif fillvar_select == 'rh':
    title = "RH"
    figtag = fillvar_select
    units_var1 = '%'
    # cmap='RdBu_r'
    cmap='BrBG'
    cmax=100; cmin=20
    cmax_diff=2; cmin_diff=-1*cmax_diff
# Absolute vorticity
elif fillvar_select == 'avor':
    title = "AVor"
    figtag = fillvar_select
    cmap='GnBu'
    units_var1 = '10$^{-6}$ /s'
    cmax=100; cmin=20
    cmax_diff=100; cmin_diff=-1*cmax_diff


### Main cross section

In [ ]:
# Loop over sensitivity tests
# for ktest in range(ntest):
for ktest in range(1):

        test_str=tests[ktest].upper()
        # fig_title = title+' ('+test_str.upper()+')'# ('+hr_tag+' h)'
        fig_title = storm.capitalize()#+' ('+test_str.upper()+')'# ('+hr_tag+' h)'
        
        # create figure
        fig, axs = plt.subplots(2, 1, figsize=(8,6), height_ratios=[.65,.35],
        # fig, axs = plt.subplots(3, 1, figsize=(8,6), height_ratios=[5,2,2],
                                layout='constrained', dpi=200)#, squeeze=True,)
        # fig.suptitle(fig_title)

        ########################################

        # MAIN PANEL

        axs[0].set_ylabel('Pressure [hPa]')

        if fillvar_select == 'lwcrf':
                pltvar=crf_binned[ktest,:,:]
                norm = colors.TwoSlopeNorm(vmin=-7, vcenter=0, vmax=5)
        elif fillvar_select == 'rh':
                qv=qv_binned[ktest,:,:]
                tmpk=tmpk_binned[ktest,:,:]
                pltvar = calc_relh(qv, pres[np.newaxis,:]*1e2, tmpk, ice=True)
                norm = colors.Normalize(vmin=cmin, vmax=cmax)

        cpltvar=cvar_binned[ktest,:,:]

        # fill contour
        im = axs[0].pcolormesh(bins[0:nbins-1], pres, np.transpose(pltvar), cmap=cmap, alpha=0.8,
                               norm=norm,zorder=2)

        fig.colorbar(im, ax=axs[0], shrink=0.75, ticks=ticker.AutoLocator(),
                     label=units_var1, pad=-0.28, extend='both')

        axs[0].set_ylim(100,np.max(pres))
        axs[0].invert_yaxis()
        axs[0].set_yscale('log')
        axs[0].set_xscale(log_x)
        axs[0].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
        axs[0].yaxis.set_minor_formatter(matplotlib.ticker.ScalarFormatter())

        # axs[0].set_xlabel(xlabel)
        xlim = (np.min(bins), np.max(bins))
        xlim = (0.4, 0.95)
        # xlim = (0.6, 0.95)
        axs[0].set_xlim(xlim)

        # Line contour
        # clevs = np.arange(lcmin, lcmax, lcint)
        clevs = [1,5,10,50,100,500,1000]
        clevs = np.concatenate((-1*np.flip(clevs),clevs))
        # cpltvar=np.gradient(binvar_c_mn,10000,axis=1,edge_order=2)*-1e5
        im = axs[0].contour(bins[0:nbins-1], pres, np.transpose(cpltvar), clevs, colors='black', zorder=2)
        axs[0].clabel(im, im.levels, inline=True, fontsize=13)

        ########################################

        # STRAT FRACTION PANEL
        subp_title = 'Cloud Classification'# ('+hr_tag+' h)'
        # axs[1].set_title(subp_title)
        axs[1].set_xlabel(xlabel)
        axs[1].set_xscale(log_x)

        # As fraction of all-points-total
        # axs[1].set_ylabel('Area fraction (*10$^3$)')
        # axs[1].set_ylabel("Frequency\n*10"+r'$^{-3}$')
        # axs[1].set_ylabel("Frequency [%]")
        axs[1].set_ylabel("Area fraction [%]")

        linewidth=1.6

        total=np.nansum(strat_binned[ktest,:,:])
        # total*=1e-3
        scale=1
        # axs[1].plot(bins[0:nbins-1], 1e2*strat_binned[ktest,:,0]/total \
        #         , ".k", label="Non-raining")
        axs[1].plot(bins[0:nbins-1], scale*1e2*strat_binned[ktest,:,1]/total \
                , "-r", label="Deep", linewidth=linewidth)
        # axs[1].plot(bins[0:nbins-1], 1e2*strat_binned[ktest,:,2]/total \
        #         , "--r", label="Cong", linewidth=2)
        # axs[1].plot(bins[0:nbins-1], 1e2*strat_binned[ktest,:,3]/total \
        #         , ":r", label="Shallow", linewidth=2)
        axs[1].plot(bins[0:nbins-1], scale*1e2*strat_binned[ktest,:,4]/total \
                , "--r", label="Strat", linewidth=linewidth)
        axs[1].plot(bins[0:nbins-1], scale*1e2*strat_binned[ktest,:,5]/total \
                , ":r", label="Anvil", linewidth=linewidth)

        # axs[1].set_yscale('log')
        # axs[1].set_ylim([0,5.5])

        axs[1].set_xlim(xlim)

        # Dummy placeholders just to include in legend
        axs[1].plot(bins[0:nbins-1], rain_binned[ktest,:]*np.nan, "-k", label=r"$P$", linewidth=linewidth)
        axs[1].plot(bins[0:nbins-1], rain_binned[ktest,:]*np.nan, "--k", label="LW-ACRE", linewidth=linewidth)
        # axs[1].plot(bins[0:nbins-1], rain_binned[ktest,:]*np.nan, "--k", label="Percent.", linewidth=2)

        # Add rain rate
        twin1 = axs[1].twinx()
        rain_hrly = rain_binned/24 # mm/d --> mm/hr
        lv0=2.5e6
        rain_wpm2 = lv0*rain_binned/(3600*24) # mm/d --> kg/m2/s
        twin1.plot(bins[0:nbins-1], rain_hrly[ktest,:], "-k", label="Rain", linewidth=linewidth)
        # twin1.plot(bins[0:nbins-1], rain_wpm2[ktest,:], "-k", label="Rain", linewidth=linewidth)
        twin1.set_ylabel(r'$P$ [mm/hr]')
        # twin1.set_ylim(0,23)
        # twin1.set_ylabel('W/m$^2$')
        # twin1.set_yscale('log')
        # twin1.set_ylim(1e0,1e5)

        axs[1].legend(frameon=False, prop={'size': 12})#, bbox_to_anchor=(0.3, 0.10))

        # Add LW-ACRE
        twin2 = axs[1].twinx()
        twin2.plot(bins[0:nbins-1], lwacre_binned[ktest,:], "--k", label="LW-ACRE", linewidth=linewidth)
        twin2.set_ylabel('LW-ACRE\n[W/m$^2$]')
        # twin2.set_yscale('log')
        # twin2.set_ylim(0,135)
        # twin2.set_ylim(1e0,1e3)

        # Set y-limits
        # axs[1].set_ylim(0,5.5)
        # Change y-axis format to scientific notation
        # axs[1].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
        # axs[2].yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter(useMathText=True))
        # axs[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter())

        # Modify axes
        sns.despine(offset=10,ax=axs[0], bottom=True)
        axs[0].set_xticks([])
        # axs[1].set_xticks([])
        sns.despine(offset=10,ax=axs[1], left=False, bottom=False, right=True, top=True)

        # Additional y-axes
        sns.despine(offset=10,ax=twin1, left=True, bottom=True, right=False, top=True)
        sns.despine(offset=10,ax=twin2, left=True, bottom=True, right=False, top=True)
        twin1.spines['right'].set_position(('outward', 5))
        twin2.spines['right'].set_position(('outward', 57))

        # plt.savefig(figdir+'csf_binned_'+storm+'.png',dpi=400, facecolor='white', \
        #             bbox_inches='tight')#, pad_inches=0.2)
        plt.savefig(figdir+'csf_binned_'+storm+'.pdf')
        plt.show()
        # figtag2 = figtag+'_'+ivar_select+fig_extra
        plt.close()